In [11]:
import os
import base64
from io import BytesIO
from PIL import Image
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [12]:
# Inicialización

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key sin configurar")

MODEL = "gpt-4o-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [13]:
system_message = "Eres un asistente útil y cortéz para hacer traducciones solo del español al inglés. "
system_message += "Debes responder siempre en espanol a excepcion cuando debas traducir al inglés."
system_message += "Se siempre preciso. Si te solicitan traducción a un idioma distinto al inglés."
system_message += "Debes responder que no estás facultado en ese idioma."

In [14]:
# Función para generar la imagen con DALL·E
def generar_imagen(prompt_traduccion):
    image_response = openai.images.generate(
        model="dall-e-3",
        prompt=f"Representación visual de: {prompt_traduccion}, en estilo digital artístico moderno.",
        size="1024x1024",
        n=1,
        response_format="b64_json",
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [15]:
def chat(history):
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    image = None

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]
    image = generar_imagen(reply)
    return history, image

In [10]:
# generar_imagen("The pretty island")

In [18]:
# Tema personalizado
tema_personal = gr.themes.Base(primary_hue="gray", font=["Arial", "sans-serif"]).set(
    body_text_color="blue",
    background_fill_primary="black",
    input_background_fill="white",
    block_background_fill="black"
)

def delete_chat_e_imagen():
    return [], None

with gr.Blocks(theme=tema_personal) as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500)
    with gr.Row():
        entry = gr.Textbox(label="Chatea con nuestro Agente de IA:")
    with gr.Row():
        clear = gr.Button("Clear")

    def do_entry(message, history):
        histppory += [{"role": "user", "content": message}]
        return "", history

    entry.submit(do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, image_output]
    )
    clear.click(
        delete_chat_e_imagen,
        inputs=None,
        outputs=[chatbot, image_output],
        queue=False
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.
